# 03b — Geração automática de Knowledge Components (KCGen-KT)

Pipeline de geração automática de KCs por assignment para o CSEDM, adaptado de Duan et al. (2025) — *KCGen-KT*. O fluxo completo (a ser implementado nas tarefas 1–7) compreende:

1. **Diversity sampling** de submissões corretas por problema (esta tarefa).
2. **Geração de KCs por LLM** a partir do código-fonte bruto (não AST — ablação de Duan et al., 2025, Table 4).
3. **Clustering Sentence-BERT + HAC** dos KCs brutos por assignment.
4. **Rotulagem dos clusters via LLM** (Table 9 de Duan et al.).
5. **Q-matrix per-assignment** (10–15 KCs).
6. **KC correctness labeling** das submissões incorretas via LLM (Table 10).
7. **Validação post-hoc com srcML** comparando KCs gerados a assinaturas AST.

**Decisões críticas (todas baseadas em Duan et al., 2025):**
- LLM recebe **código bruto** — NÃO AST (Table 4: AST-only piora AUC de 0.812 → 0.784).
- **5 submissões por problema** via stratified sampling por número de tentativas (Table 5 — n=5 é o ponto ótimo).
- Uso exclusivo de `Release/Train` (mesma população de Shi et al., 2022) para evitar data leakage.
- **Cache obrigatório** em todas as etapas LLM — notebook deve ser re-executável sem novas chamadas API.

**Modelo LLM:** `claude-haiku-4-5-20251001` (custo total estimado: ~\$39–40).

In [1]:
# Setup — imports, SEED, paths
import json
import pickle
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# Bibliotecas necessárias para as etapas posteriores do pipeline KCGen-KT
import anthropic                              # Etapas 2, 4 e 6 — chamadas LLM
import sentence_transformers                  # Etapa 3 — embeddings dos KCs
import scipy                                  # Etapa 3 — HAC clustering
import sklearn                                # Etapa 3 — métricas auxiliares

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data" / "CSEDM"
RESULTS_ROOT = REPO_ROOT / "results"
RESULTS_ROOT.mkdir(exist_ok=True)

SEQUENCES_PATH = RESULTS_ROOT / "sequences_bkt_dkt.pkl"
CODE_STATES_PATH = DATA_ROOT / "Release" / "Train" / "Data" / "CodeStates" / "CodeStates.csv"

ASSIGNMENTS = [439, 487, 492, 494, 502]
LLM_MODEL = "claude-haiku-4-5-20251001"

print(f"SEED        = {SEED}")
print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"sequences   = {SEQUENCES_PATH.exists()} ({SEQUENCES_PATH.name})")
print(f"code states = {CODE_STATES_PATH.exists()} ({CODE_STATES_PATH.name})")
print(f"anthropic   = {anthropic.__version__}")
print(f"s-transformers = {sentence_transformers.__version__}")
print(f"scipy       = {scipy.__version__}")
print(f"sklearn     = {sklearn.__version__}")

/home/leokuntz/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED        = 42
REPO_ROOT   = /home/leokuntz/Documents/repositories/studies/tcc.edm.kt
sequences   = True (sequences_bkt_dkt.pkl)
code states = True (CodeStates.csv)
anthropic   = 0.99.0
s-transformers = 5.4.1
scipy       = 1.17.1
sklearn     = 1.8.0


### 1.1 — Carregamento e diversity sampling de submissões corretas

**Contexto:** A geração de KCs via LLM exige um conjunto pequeno e diverso de submissões corretas para cada problema. Como o CSEDM não fornece enunciados textuais dos problemas, o LLM deve inferir o conhecimento exigido a partir das soluções dos próprios estudantes — soluções demasiado parecidas levariam o modelo a sub-representar abordagens alternativas (e, portanto, KCs).

**Hipótese:** Estratificar as submissões corretas pelo número total de tentativas até o acerto (proxy para diversidade de estratégia: "acertou direto" vs "acertou após várias tentativas") permitirá amostrar até 5 submissões com cobertura ampla de abordagens, mantendo a fração ótima de Duan et al. (2025, Table 5) com custo proporcional.

**Referência:** Duan et al. (2025), *Automated Knowledge Component Generation for Interpretable Knowledge Tracing in Coding Problems* — Table 5 (ablação `n_submissions`: 1=0.798, 3=0.807, **5=0.812**, 7=0.811, 10=0.810). A estratificação por número de tentativas é a heurística adotada neste trabalho para garantir diversidade dentro do limite de n=5; Duan et al. amostram aleatoriamente, mas mencionam que a representatividade do conjunto importa para a qualidade dos KCs.

In [2]:
def load_correct_samples(sequences_path: Path, code_states_path: Path) -> dict:
    """Carrega submissões corretas do Release/Train com código Java associado.

    Para cada par (estudante, problema), identifica a primeira submissão correta
    cronologicamente, conta as tentativas anteriores (eventos do mesmo estudante
    no mesmo problema antes do acerto) e faz join com CodeStates.csv via
    CodeStateID para recuperar o código Java efetivamente submetido.

    Parameters
    ----------
    sequences_path : Path
        Caminho para results/sequences_bkt_dkt.pkl (artefato do notebook 02).
    code_states_path : Path
        Caminho para data/CSEDM/Release/Train/Data/CodeStates/CodeStates.csv.

    Returns
    -------
    dict
        {assignment_id: {problem_id: [
            {'subject_id': str, 'codestate_id': str, 'n_attempts_before': int,
             'total_attempts': int, 'code': str},
            ...
        ]}}
        Apenas a primeira submissão correta por (estudante, problema) é incluída.
    """
    with open(sequences_path, "rb") as f:
        artifact = pickle.load(f)

    code_states = pd.read_csv(code_states_path)
    code_map = dict(zip(code_states["CodeStateID"], code_states["Code"]))

    correct_by_assignment = {}
    for assignment_id, sequences in artifact["train"].items():
        per_problem: dict[int, list[dict]] = {}
        for seq in sequences:
            events = seq["events"].sort_values("ServerTimestamp", kind="stable")
            for problem_id, problem_events in events.groupby("ProblemID", sort=False):
                problem_events = problem_events.sort_values("ServerTimestamp", kind="stable")
                correct_mask = problem_events["correct"] == 1
                if not correct_mask.any():
                    continue
                first_correct_pos = correct_mask.values.argmax()
                first_correct = problem_events.iloc[first_correct_pos]
                code = code_map.get(first_correct["CodeStateID"])
                if code is None or (isinstance(code, float) and pd.isna(code)):
                    continue
                n_before = int(first_correct_pos)
                per_problem.setdefault(int(problem_id), []).append({
                    "subject_id": str(seq["subject_id"]),
                    "codestate_id": str(first_correct["CodeStateID"]),
                    "n_attempts_before": n_before,
                    "total_attempts": n_before + 1,
                    "code": str(code),
                })
        correct_by_assignment[int(assignment_id)] = per_problem

    return correct_by_assignment


def _bucket_for(total_attempts: int) -> int:
    """Mapeia número total de tentativas em um dos 5 buckets de diversidade.

    Buckets (Duan et al., 2025, motivação: diversidade de estratégia):
      1 → acertou na 1ª tentativa
      2 → 2–3 tentativas
      3 → 4–6 tentativas
      4 → 7–10 tentativas
      5 → >10 tentativas
    """
    if total_attempts <= 1:
        return 1
    if total_attempts <= 3:
        return 2
    if total_attempts <= 6:
        return 3
    if total_attempts <= 10:
        return 4
    return 5


def diversity_sample(correct_events: list[dict], n: int = 5,
                     rng: random.Random | None = None) -> list[dict]:
    """Estratifica as submissões corretas em 5 buckets e amostra até n.

    Para cada bucket disponível (na ordem 1..5), seleciona aleatoriamente uma
    submissão; retorna no máximo n=5 submissões. Buckets vazios são pulados.

    Parameters
    ----------
    correct_events : list[dict]
        Lista de submissões corretas para um par (assignment, problem) — cada
        item deve ter as chaves 'total_attempts' e 'code'.
    n : int
        Número máximo de submissões a retornar. Padrão 5 (Duan et al., 2025,
        Table 5 — ponto ótimo).
    rng : random.Random | None
        Gerador de aleatórios para reproduzibilidade. Se None, usa o gerador
        global semeado com SEED.

    Returns
    -------
    list[dict]
        Até n submissões, ordenadas pelo bucket (estratégia de solução direta
        primeiro, com mais tentativas por último).
    """
    if rng is None:
        rng = random.Random(SEED)

    buckets: dict[int, list[dict]] = {1: [], 2: [], 3: [], 4: [], 5: []}
    for event in correct_events:
        b = _bucket_for(int(event["total_attempts"]))
        buckets[b].append(event)

    sampled = []
    for b in (1, 2, 3, 4, 5):
        if len(sampled) >= n:
            break
        if buckets[b]:
            sampled.append(rng.choice(buckets[b]))
    return sampled


# Carrega submissões corretas e suas representações em código Java.
correct_by_assignment = load_correct_samples(SEQUENCES_PATH, CODE_STATES_PATH)

# Aplica diversity sampling por (assignment, problem) — RNG semeado para reproduzibilidade.
rng = random.Random(SEED)
sampled_codes: dict[int, dict[int, list[str]]] = {}
sampling_stats: list[dict] = []

for assignment_id in ASSIGNMENTS:
    sampled_codes[assignment_id] = {}
    per_problem = correct_by_assignment.get(assignment_id, {})
    for problem_id, events in per_problem.items():
        sampled = diversity_sample(events, n=5, rng=rng)
        sampled_codes[assignment_id][problem_id] = [s["code"] for s in sampled]
        bucket_counts = Counter(_bucket_for(s["total_attempts"]) for s in sampled)
        sampling_stats.append({
            "AssignmentID": assignment_id,
            "ProblemID": problem_id,
            "n_correct_total": len(events),
            "n_sampled": len(sampled),
            "buckets_used": sorted(bucket_counts.keys()),
        })

stats_df = pd.DataFrame(sampling_stats).sort_values(["AssignmentID", "ProblemID"]).reset_index(drop=True)

summary = (
    stats_df.groupby("AssignmentID")
    .agg(
        n_problems=("ProblemID", "nunique"),
        mean_samples=("n_sampled", "mean"),
        min_samples=("n_sampled", "min"),
        max_samples=("n_sampled", "max"),
        mean_correct_pool=("n_correct_total", "mean"),
    )
    .round(2)
)

print("Resumo do diversity sampling por assignment:")
print(summary.to_string())
print()
print(f"Total de problemas cobertos: {stats_df['ProblemID'].count()} "
      f"(soma de problemas únicos por assignment, com possível overlap entre assignments).")
print(f"Média global de amostras/problema: {stats_df['n_sampled'].mean():.2f}")
print(f"Distribuição do número de amostras por problema: "
      f"{dict(Counter(stats_df['n_sampled']).most_common())}")

Resumo do diversity sampling por assignment:
              n_problems  mean_samples  min_samples  max_samples  mean_correct_pool
AssignmentID                                                                       
439                   10           4.9            4            5              216.6
487                   10           4.9            4            5              177.3
492                   10           5.0            5            5              176.9
494                   10           5.0            5            5              180.1
502                   10           4.9            4            5              184.5

Total de problemas cobertos: 50 (soma de problemas únicos por assignment, com possível overlap entre assignments).
Média global de amostras/problema: 4.94
Distribuição do número de amostras por problema: {5: 47, 4: 3}


**Achado:** Os 5 assignments do Release/Train (A439, A487, A492, A494, A502) são processados em sua totalidade — `n_problems` por assignment está reportado em `summary`. A média de amostras/problema é ≤5 e tipicamente próxima de 5 nos problemas com maior pool de submissões corretas; problemas raros, com poucos acertos, geram menos amostras (1 por bucket disponível). A distribuição de `n_sampled` está concentrada em 5 — comportamento esperado para uma turma de ~233 estudantes em problemas de granularidade média.

**Implicação para modelagem:** O dicionário `sampled_codes` é o input direto da Etapa 2 (geração de KCs por LLM). Mantemos `n=5` porque Duan et al. (2025, Table 5) demonstram saturação acima desse número (AUC de 0.811 com n=7 e 0.810 com n=10 vs 0.812 com n=5) — aumentar `n` adicionaria custo LLM sem ganho mensurável na qualidade dos KCs. A estratificação por tentativas garante que problemas com soluções fáceis (todos acertam de primeira) ainda sejam representados, mesmo quando o pool de buckets 2–5 é vazio. Esta escolha é uma decisão arquitetural local: Duan et al. amostram aleatoriamente; nossa estratégia prioriza diversidade explícita de estratégia, hipótese a ser validada na Etapa 7 (post-hoc, via assinaturas srcML).